# Train Vesper's wake words on Colab

This trains small **acoustic wake-word models** with [livekit-wakeword](https://github.com/livekit/livekit-wakeword) (Apache-2.0):

| file | what it does in the assistant |
|---|---|
| `vesper.onnx` | the name: speech-to-text only runs after it fires at the start of what you say, so saying "Vesper" to chat mid-sentence no longer wakes it |
| `stop.onnx` | stops a spoken reply the instant you say "stop" |
| `clip_that.onnx` (optional) | runs "clip that" the moment it's heard: no speech-to-text, no Claude |

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
It is mostly unattended: about **1.5–2 h** for the name with the *standard* preset, plus about an hour per extra phrase.
Keep the tab open (free Colab disconnects closed or idle tabs). Finished models are copied to Google Drive as they complete, so a disconnect doesn't lose them.

Training uses only synthetic voices (Piper TTS, ~900 speakers blended) plus public noise and speech datasets. None of your audio is uploaded.

## 1 · Check the GPU

In [ ]:
!nvidia-smi -L || echo "No GPU. Runtime -> Change runtime type -> T4 GPU, then run again."

## 2 · Install the trainer (~2 min)

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y espeak-ng libsndfile1 ffmpeg sox > /dev/null
!pip -q install "livekit-wakeword[train,eval,export]>=0.2"
!livekit-wakeword --help | head -5

## 3 · Choose what to train

In [ ]:
# @title Phrases and quality
NAME = "vesper"  # @param {type:"string"}
TRAIN_STOP = True  # @param {type:"boolean"}
EXTRA_PHRASE = ""  # @param {type:"string"}
QUALITY = "standard"  # @param ["quick", "standard", "best"]
SAVE_TO_DRIVE = True  # @param {type:"boolean"}

# Words that sound close but must NOT fire. Edit if your name isn't Vesper.
NAME_NEGATIVES = ["whisper", "vespa", "desperate", "esper", "hesper", "best per", "less per", "vest pair",
                  "prosper", "vapor", "viper", "pepper", "vesting", "yes sir", "west pier"]
STOP_NEGATIVES = ["top", "shop", "stock", "stomp", "step", "stuff", "spot", "strap", "stopwatch",
                  "non stop", "bus stop", "it stopped", "stop the stream"]

In [ ]:
import json, os, re, yaml

DATA = "/content/lkww-data"       # shared downloads (datasets, TTS voices)
OUT = "/content/output"
PRESETS = {  # samples per class, training steps, model size
    "quick":    dict(n_samples=3000,  n_samples_val=600,  steps=10000, size="small"),
    "standard": dict(n_samples=10000, n_samples_val=2000, steps=30000, size="small"),
    "best":     dict(n_samples=25000, n_samples_val=5000, steps=60000, size="medium"),
}
p = PRESETS[QUALITY]
slug = lambda s: re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")

def config(model_name, phrases, negatives):
    return {
        "model_name": model_name, "target_phrases": phrases, "custom_negative_phrases": negatives,
        "n_samples": p["n_samples"], "n_samples_val": p["n_samples_val"],
        "n_background_samples": 1000, "n_background_samples_val": 250, "tts_batch_size": 50,
        "data_dir": DATA, "output_dir": OUT,
        "augmentation": {"clip_duration": 2.0, "batch_size": 16, "rounds": 2,
                         "background_paths": [f"{DATA}/backgrounds"], "rir_paths": [f"{DATA}/rirs"]},
        "model": {"model_type": "conv_attention", "model_size": p["size"]},
        "steps": p["steps"], "learning_rate": 1e-4, "max_negative_weight": 2000, "target_fp_per_hour": 0.2,
    }

name = slug(NAME)
jobs = [config(name, [NAME.lower(), f"hey {NAME.lower()}"], NAME_NEGATIVES)]
if TRAIN_STOP:
    jobs.append(config("stop", ["stop", "stop it", "okay stop"], STOP_NEGATIVES))
if EXTRA_PHRASE.strip():
    jobs.append(config(slug(EXTRA_PHRASE), [EXTRA_PHRASE.lower().strip()], []))

from livekit.wakeword import WakeWordConfig
for job in jobs:
    WakeWordConfig(**job)  # fail now, not after an hour, if a field is wrong
    with open(f"/content/{job['model_name']}.yaml", "w") as fh:
        yaml.safe_dump(job, fh, sort_keys=False)
print("Will train:", ", ".join(j["model_name"] for j in jobs), f"({QUALITY})")

## 4 · Download the training data (~17 GB, ~15 min, once per session)

In [ ]:
!livekit-wakeword setup --config /content/{jobs[0]['model_name']}.yaml

## 5 · Train (the long part)

In [ ]:
import shutil
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE = "/content/drive/MyDrive/vesper-wake-words"
    os.makedirs(SAVE, exist_ok=True)

for job in jobs:
    n = job["model_name"]
    print(f"\n===== {n}: generate -> augment -> features -> train -> export -> evaluate =====")
    get_ipython().system(f"livekit-wakeword run /content/{n}.yaml")
    if SAVE_TO_DRIVE:
        for suffix in (".onnx", "_eval.json", "_det.png"):
            src = f"{OUT}/{n}/{n}{suffix}"
            if os.path.exists(src):
                shutil.copy(src, SAVE)
        print(f"Saved {n} to Drive: {SAVE}")

## 6 · Results and download

In [ ]:
from IPython.display import Image, display

print(f"{'model':12s} {'recall':>7s} {'false/hr':>9s} {'threshold':>10s}")
for job in jobs:
    n = job["model_name"]
    path = f"{OUT}/{n}/{n}_eval.json"
    if not os.path.exists(path):
        print(f"{n:12s} not finished (check the log above)")
        continue
    m = json.load(open(path))
    print(f"{n:12s} {m.get('optimal_recall', m['recall']):7.1%} {m.get('optimal_fpph', m['fpph']):9.2f} "
          f"{m['optimal_threshold']:10.2f}")
    if os.path.exists(f"{OUT}/{n}/{n}_det.png"):
        display(Image(f"{OUT}/{n}/{n}_det.png", width=420))

print("\nIn the assistant: Setup tab -> Trigger words -> Add trained model -> pick the .onnx file,")
print("and type the threshold from the table above when it asks.")

In [ ]:
from google.colab import files
for job in jobs:
    n = job["model_name"]
    if os.path.exists(f"{OUT}/{n}/{n}.onnx"):
        files.download(f"{OUT}/{n}/{n}.onnx")

### If it's not right
* **Misses you often:** in the assistant lower that model's threshold by 0.05–0.1 (Setup tab → remove and re-add it), or retrain with `QUALITY = "best"`.
* **Fires on its own:** add the words it fires on to `NAME_NEGATIVES` and retrain, or raise the threshold.
* **Disconnected mid-run:** run the setup and training cells again. Models already saved to Drive don't need retraining; drop them from `jobs` first.